# Modello di Black-Litterman

Il modello di Black-Litterman risolve la sensibilità della MVO partendo dal Market Equilibrium. Invece di stimare i rendimenti attesi dai dati storici, li "inverte" dai pesi di mercato esistenti.Il framework matematicoIl vettore dei rendimenti attesi combinati $\mu_{BL}$ è dato da:$$\mu_{BL} = [(\tau \Sigma)^{-1} + P^T \Omega^{-1} P]^{-1} [(\tau \Sigma)^{-1} \Pi + P^T \Omega^{-1} Q]$$$\Pi$: Rendimenti di equilibrio (Implied Returns).$\tau$: Scalare che riflette l'incertezza del prior (solitamente tra 0.025 e 0.05).$P$: Matrice di "picking" che identifica gli asset oggetto delle view.$Q$: Vettore dei rendimenti attesi per le specifiche view.$\Omega$: Matrice di incertezza delle view.Implementazione in PythonQuesta funzione calcola i rendimenti attesi e la covarianza posteriore secondo la logica di asset allocation professionale.

In [1]:
def black_litterman_analysis(cov_matrix, market_weights, risk_aversion, views, view_confidences):
    """
    Sviluppo del modello Black-Litterman.
    - cov_matrix: Matrice di covarianza (preferibilmente shrunk).
    - market_weights: Pesi di capitalizzazione di mercato.
    - risk_aversion: Coefficiente gamma (es. 3.0).
    - views: Dizionario {asset_index: expected_return}.
    """
    # 1. Calcolo dei rendimenti impliciti (Equilibrium Prior)
    # Pi = lambda * Sigma * w
    pi = risk_aversion * cov_matrix.dot(market_weights)

    # 2. Setup delle Views (P, Q, Omega)
    n_assets = len(market_weights)
    n_views = len(views)
    P = np.zeros((n_views, n_assets))
    Q = np.array(list(views.values()))

    for i, asset_idx in enumerate(views.keys()):
        P[i, asset_idx] = 1

    # Parametro tau e matrice Omega (incertezza delle view)
    tau = 0.05
    # Omega diagonale basata sull'incertezza fornita (es. varianza del prior)
    omega = np.diag(np.diag(P.dot(tau * cov_matrix).dot(P.T)))

    # 3. Calcolo del Posterior Expected Returns
    term1 = np.linalg.inv(np.linalg.inv(tau * cov_matrix) + P.T.dot(np.linalg.inv(omega)).dot(P))
    term2 = np.linalg.inv(tau * cov_matrix).dot(pi) + P.T.dot(np.linalg.inv(omega)).dot(Q)

    mu_bl = term1.dot(term2)

    return mu_bl

# Esempio: View rialzista sugli USA (indice 0) al 12%
# bl_returns = black_litterman_analysis(cov_matrix, mkt_w, 3.0, {0: 0.12}, [0.1])